# NCBI Virus Dataset Creation

Author: Alexander Maksiaev

Purpose: Create dataset using NCBI, by de-duplicating from Andersen/GISAID, and relabeling sequences. 

In [1]:
# Housekeeping

import os
import pandas as pd
import dateutil
import re
import importlib
import utils  
importlib.reload(utils)
from utils import * 

In [5]:
# Paths

# Dates
start_date = "06-28-2025"
end_date = "07-04-2025"
date_range = start_date + "--" + end_date
# update_date = "06-17-2025"
# genotypes = ["D1.1", "B3.2", "B3.6", "B3.13", "A3", "A2", "C2.1"]

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
downloads = home + "NCBI_Virus/downloads/" + date_range + "_Antarctica_North_America_South_America/"
temp_files = home + "NCBI_Virus/temp/"
complete_files = home + "NCBI_Virus/complete/" + date_range + "_Antarctica_North_America_South_America/" # + "_B3_13/"
# gisaid_andersen = home + "Cats/Datasets/GISAID_Andersen/GISAID_Andersen_cats_only_6_5_2025/"
# combined_files = home + "Cats/Datasets/GISAID_Andersen_NCBI_Virus/cat_genotypes_" + update_date + "/" # 04-14-2025--06-05-2025"
# gisaid_ant = home + "GISAID/complete/" + date_range + "_all_genotypes_ant/"
# gisaid_southa = home + "GISAID/complete/" + date_range + "_all_genotypes_southa/"
# gisaid_ant_southa = home + "GISAID/complete/" + date_range + "_all_genotypes_Antarctica_South_America"

gisaid_andersen = home + "Combinations/GISAID_Andersen/B3_13/" + date_range + "_B3_13_Antarctica_North_America_South_America/" # B3_13/"
combined_files = home + "Combinations/GISAID_Andersen_NCBI_Virus/" + date_range + "_Antarctica_North_America_South_America/" # + update_date + "_B3_13/"
references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references/"
# references = "C:/Users/maksi/Documents/Statistics/projects/Avian_Flu/references/"

os.chdir(references)
states_ref = pd.read_csv("states_ref.csv")
# genotypes_df = pd.read_excel("genotype_key.xlsx")

# genotypes = list(genotypes_df["Genotype"])

genotypes = ["B3.13", "D1.1"]

os.chdir(downloads)

## De-Duplication

In [6]:
metadata = pd.read_csv("sequences.csv")

print(len(metadata))

# Make sure we only have completed sequences -- 8 segments each 

metadata_counts = metadata.groupby(metadata.SRA_Accession, as_index=False).size()

print(metadata_counts)

metadata_counted = metadata.merge(metadata_counts, on="SRA_Accession")

# Only keep those with size=8

metadata_complete_segs = metadata_counted[metadata_counted["size"] >= 8] # May have duplicates

metadata_complete_segs = metadata_complete_segs.drop_duplicates(subset="GenBank_Title", keep="first") # Get rid of duplicate segments

# Now only accept == 8 segments

metadata_segments = metadata_complete_segs[metadata_complete_segs["size"] == 8]

metadata_segments

174
Empty DataFrame
Columns: [SRA_Accession, size]
Index: []


,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genotype,Isolate,...,Length,Nuc_Completeness,Geo_Location,Host,Tissue_Specimen_Source,Submitters,Collection_Date,Release_Date,Molecule_type,size


In [14]:
# De-duplicate from Andersen/GISAID using isolate

# Function to prepare dataframes
def fasta_df_complete(file_name, state_ref):

    fasta = pd.DataFrame()
    headers = []
    isolate_ids = []
    isolate_names = []
    subtypes = []
    # segments = []
    collection_dates = []
    sequences = []
    host_types = []
    species = []
    identifiers = []
    genotypes = []
    with open(file_name) as f:
        lines = f.readlines()
        for num, line in enumerate(lines):
            # print(line)
            if line[0] == ">": # If it's a header
                if line[1:].strip() not in headers: # And the previous line is not a header we've seen before
                    header = line[1:].strip() # Remove the ">"
                    # print(header)
                    try:
                        split_header = header.split("|")
                        if len(header.split("|")) > 6:
                            identifier = header.split("|")[0]
                            identifiers.append(identifier)
                            split_first_header = split_header[1].split("/")
                        else:
                            identifiers.append("unknown")
                            split_first_header = split_header[0].split("/")
                        # print(split_first_header)
                        # print(split_header)
                        headers.append(header) 
                        isolate_ids.append(split_first_header[3])
                        isolate_names.append(split_header[-6]) # We'll need to extract data from this too
                        # print(split_header[2].split("_")[-1])
                        subtypes.append(split_header[-5].split("_")[-1])  # Get only H5N1
                        genotypes.append(split_header[-1])
                        # segments.append(split_header[-2]) # .split("|")[-1])
                        host_types.append(split_header[-2])
                        species.append(split_first_header[1])
                        # if split_header[4] == "2024-01-01":
                        #     collection_dates.append("2024") # No samples were collected 1/1/2024, these are all unknown 
                        # elif split_header[4] == "2025-01-01":
                        #     collection_dates.append("2025")
                        # else: 
                        collection_dates.append(split_header[-3])
                        # collection_dates.append(split_header[-1].split("_")[-1])
                        if num < len(lines): # If we're not at the last line
                            # for i, l in enumerate(lines[num + 1:]):
                            i = num
                            sequence = ""
                            # print(lines[i])
                            # print(lines[i + 1])
                            while i < len(lines) - 1 and lines[i + 1][0] != ">": # While the next line is part of a sequence
                                sequence = sequence + lines[i + 1].strip()
                                i += 1
                            sequences.append(sequence) # Add next line to sequences
                    except:
                        headers.remove(header)
                        identifiers.remove(identifier)
                        print(header)
                        continue
        f.close()

    # Create columns for data frame 
    fasta["Header"] = headers
    fasta["Isolate_Id"] = isolate_ids
    fasta["Isolate_Name"] = isolate_names
    fasta["Subtype"] = subtypes
    # fasta["Segment"] = segments
    # Geo_Location is more complicated
    fasta["Geo_Location"] = fasta["Header"].apply(lambda x: state_ref.loc[state_ref["Abbreviation"] == x.split("/")[2], 'Country'].iloc[0] + "-" + x.split("/")[2] if x.split("/")[2] in state_ref["Abbreviation"].values else state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Country'].iloc[0] + "-" + state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Abbreviation'].iloc[0] if x.split("/")[2].replace("_", " ") in state_ref["State"].values else x.split("/")[2].replace(": ", "-"))
    fasta["Date Collected"] = collection_dates
    fasta["Species"] = species
    fasta["Host_Type"] = host_types
    fasta["Genotype"] = genotypes
    fasta["Sequence"] = sequences
    # if len(identifiers) == len(fasta):
    fasta["Identifier"] = identifiers
    
    return fasta

# If even one isolate in this list exists in the NCBI Virus dataframe, remove it from NCBI Virus dataframe
gisaid_isolates = []
andersen_isolates = []
# Grab files
# for genotype in genotypes:
    # print(gisaid_andersen + genotype.replace(".", "_") + "/")
    # for dirpath, dirs, files in os.walk(gisaid_andersen + genotype.replace(".", "_") + "/" + date_range + "_" + genotype.replace(".", "_") + "/"):
for dirpath, dirs, files in os.walk(gisaid_andersen):
    for file in files:
        file_name = os.path.join(dirpath, file)
        # print(file_name)
        if ".fasta" in file_name:
            fasta_file = fasta_df_complete(file_name, states_ref) # Convert fasta file to dataframe
            print(fasta_file)
            # isolates = fasta_file["Isolate_Name"].apply(lambda x: x.split("/")[3])
            isolates = fasta_file["Isolate_Id"]
            for index, value in isolates.items():
                gisaid_isolates.append(value)

            isolate_ids = fasta_file["Identifier"]
            for index, value in isolates.items():
                if "SRR" in value:
                    andersen_isolates.append(value)
            
        # break 

    # for dirpath, dirs, files in os.walk(gisaid_southa):
    #     for file in files:
    #         file_name = os.path.join(dirpath, file)
    #         # print(file_name)
    #         if ".fasta" in file_name:
    #             fasta_file = fasta_df(file_name, states_ref) # Convert fasta file to dataframe
    #             print(fasta_file)
    #             # isolates = fasta_file["Isolate_Name"].apply(lambda x: x.split("/")[3])
    #             isolates = fasta_file["Isolate_Id"]
    #             for index, value in isolates.items():
    #                 gisaid_isolates.append(value)

    #             isolate_ids = fasta_file["Identifier"]
    #             for index, value in isolates.items():
    #                 if "SRR" in value:
    #                     andersen_isolates.append(value)
            
    #     break 

gisaid_isolates = list(set(gisaid_isolates))

print(len(gisaid_isolates))
print(gisaid_isolates)

                                              Header  Isolate_Id  \
0  EPI_ISL_20055135|A/red_fox/USA/028269-001/2024...  028269-001   

                    Isolate_Name Subtype Geo_Location Date Collected  Species  \
0  A/red_fox/USA/028269-001/2024    H5N1          USA     2024-01-01  red_fox   

      Host_Type Genotype                                           Sequence  \
0  other_mammal       A3  atggagaacatagtacttcttcttgcaataattagccttgttaaaa...   

         Identifier  
0  EPI_ISL_20055135  
                                              Header  Isolate_Id  \
0  EPI_ISL_20055135|A/red_fox/USA/028269-001/2024...  028269-001   

                    Isolate_Name Subtype Geo_Location Date Collected  Species  \
0  A/red_fox/USA/028269-001/2024    H5N1          USA     2024-01-01  red_fox   

      Host_Type Genotype                                           Sequence  \
0  other_mammal       A3  atgagtcttctaaccgaggtcgaaacgtacgttctctctatcgtcc...   

         Identifier  
0  EPI_ISL_20055

In [15]:
# Remove duplicates from GISAID/Andersen

print(len(metadata_segments))

count = 0
isolate_partial = {}
for value in metadata_segments["Isolate"].values:
    # partial = value.split("_")[-1] # If 25_, get the last bit
    # print(partial)
    if value != value: # If NaN
        value = ""
    digits = value.split("-")
    isolate = ""
    # other = ""
    for d in digits:
        # print(d)
        if len(d) == 6 and d.isnumeric(): # If it's just digits and not one of those weird isolates
            isolate = d + "-"
        elif len(d) == 3 and d.isnumeric():
            isolate = isolate + d
        # elif d.isnumeric() == False: # If it's a weird isolate
        #     other = d + "-"
        # else: # If it's a weird isolate
        #     other = other + d
    # Now add to list to check in Andersen files without doing wild for loops
    if len(isolate) == 10: # If this is a correctly formatted isolate
        # isolates.append(isolate)
        # All headers are followed by sequences
        isolate_partial[value] = isolate
    else: # If this is some other isolate
        isolate_partial[value] = value
    # print(isolate)
# for index, value in metadata_segments["Isolate"].items():




to_remove = andersen_isolates
for value in isolate_partial.keys():
    isolate = isolate_partial[value]
    # print(isolate)
    r = re.compile(".*" + isolate + ".*")
    # print(r)
    if len(list(filter(r.search, gisaid_isolates))) > 0:
        for i in list(filter(r.search, gisaid_isolates)):
            to_remove.append(value) 
    # print(to_remove)
    # if isolate_partial[value] in gisaid_andersen_isolates:
    # if value in gisaid_andersen_isolates:
    #     count += 1
    #     to_remove.append(value)
            # metadata_segments = metadata_segments.drop(index)

print(to_remove)
print(len(to_remove))

for value in to_remove:
    if "SRR" in value:
        metadata_segments = metadata_segments[metadata_segments["SRA_Accession"] != value]
    # print(value)
    # print(isolate_partial[value])
    else:
        metadata_segments = metadata_segments[metadata_segments["Isolate"] != value]

print(len(metadata_segments))
print(len(isolate_partial))
metadata_segments
# print(count)


2376
['24-034840-002-original-repeat2', '25-004922-001-original', '25-004922-002-original', '25-004922-003-original', '25-004922-005-original', '25-004922-006-original', '25-014338-001-original', '25-014293-001-original-repeat', '25-012798-007-original-repeat', '24-024710-002-original-repeat2', '24-024710-004-original-repeat2', '24-036182-001-original-repeat2', '24-036182-002-original-repeat2', '25-000462-001-original', '25-000462-002-original', '25-000462-003-original', '25-000462-004-original', '25-007985-001-original-repeat2', '25-011565-001-original-repeat2']
19
2224
297


,Accession,GenBank_RefSeq,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genotype,Isolate,Segment,...,Length,Nuc_Completeness,Geo_Location,Host,Tissue_Specimen_Source,Submitters,Collection_Date,Release_Date,Molecule_type,size
0,PV845802.1,GenBank,SRR33681813,SAMN48710861,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,H5N1,24-028985-001-tile,1,...,2273,partial,USA: CA,Bos taurus,mammalian milk,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",2024-10-01,2025-06-27,ssRNA(-),8
1,PV845803.1,GenBank,SRR33681813,SAMN48710861,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,H5N1,24-028985-001-tile,2,...,2253,partial,USA: CA,Bos taurus,mammalian milk,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",2024-10-01,2025-06-27,ssRNA(-),8
2,PV845804.1,GenBank,SRR33681813,SAMN48710861,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,H5N1,24-028985-001-tile,3,...,2151,partial,USA: CA,Bos taurus,mammalian milk,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",2024-10-01,2025-06-27,ssRNA(-),8
3,PV845805.1,GenBank,SRR33681813,SAMN48710861,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,H5N1,24-028985-001-tile,4,...,1704,partial,USA: CA,Bos taurus,mammalian milk,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",2024-10-01,2025-06-27,ssRNA(-),8
4,PV845806.1,GenBank,SRR33681813,SAMN48710861,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,H5N1,24-028985-001-tile,5,...,1497,partial,USA: CA,Bos taurus,mammalian milk,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",2024-10-01,2025-06-27,ssRNA(-),8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2371,PV792621.1,GenBank,SRR33476664,SAMN48398150,PRJNA980729,Influenza A virus,Alphainfluenzavirus influenzae,H5N1,25-013275-010-original,4,...,1704,partial,USA: NY,Gallus gallus,oronasopharynx,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",2025-04-22,2025-06-16,ssRNA(-),8
2372,PV792622.1,GenBank,SRR33476664,SAMN48398150,PRJNA980729,Influenza A virus,Alphainfluenzavirus influenzae,H5N1,25-013275-010-original,5,...,1497,partial,USA: NY,Gallus gallus,oronasopharynx,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",2025-04-22,2025-06-16,ssRNA(-),8
2373,PV792623.1,GenBank,SRR33476664,SAMN48398150,PRJNA980729,Influenza A virus,Alphainfluenzavirus influenzae,H5N1,25-013275-010-original,6,...,1410,partial,USA: NY,Gallus gallus,oronasopharynx,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",2025-04-22,2025-06-16,ssRNA(-),8
2374,PV792624.1,GenBank,SRR33476664,SAMN48398150,PRJNA980729,Influenza A virus,Alphainfluenzavirus influenzae,H5N1,25-013275-010-original,7,...,982,partial,USA: NY,Gallus gallus,oronasopharynx,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",2025-04-22,2025-06-16,ssRNA(-),8


In [16]:
# NCBI Virus Naming Convention Example:
# Influenza A virus |USA: IN|25-006338-001-original|H5N1|2025-02-20|Meleagris gallopavo|GenBank|SRR33124721|SAMN47941411|PRJNA980729|Influenza A virus (A/Turkey/IN/25-006338-001-original/2025(H5N1)) segment 1 polymerase PB2 (PB2) gene, complete cds
# Organism_Name | Geo_Location | Isolate | Genotype | Collection_Date | Host | GenBank_RefSeq | SRA_Accession | BioSample | BioProject | GenBank_Title


# print(metadata_segments["GenBank_Title"].iloc[0])

# Get sequences and headers together
headers = []
isolates = []
sras = []
headers_seqs = {}
with open("sequences.fasta") as f:
    lines = f.readlines()
    for num, line in enumerate(lines):
        if line[0] == ">": # If this is a header
            if line.strip() not in headers: # And is not a header we've seen before
                header = line.strip() 
                # print(header)
                split_header = header.split("|")
                # print(split_header)
                headers.append(header) 
                if num < len(lines): # If we're not at the last line
                    # for i, l in enumerate(lines[num + 1:]):
                    i = num
                    sequence = ""
                    # print(lines[i])
                    # print(lines[i + 1])
                    while i < len(lines) - 1 and lines[i + 1][0] != ">": # While the next line is part of a sequence
                        sequence = sequence + lines[i + 1].strip()
                        i += 1
                    headers_seqs[header] = sequence # Add next lines to sequences
                # name = split_header[-1] # Last part gives GenBank name, with segment and isolate
                isolate = split_header[2]
                isolates.append(isolate)
                sra = split_header[7]
                sras.append(sra)
    f.close()



# print(mask_all.all(axis=1))

In [17]:
# Double-check de-duplication
mask = metadata_segments["Isolate"].isin(isolates) # Check if isolate is in fasta
# mask_sra = metadata_segments["SRA_Accession"].isin(sras) # Check if sra is in fasta

metadata_masked = metadata_segments[mask] # Some do not have isolates, so check sra too
# metadata_masked_sra = metadata_segments[mask_sra]

# merged_metadata = metadata_masked.merge(metadata_masked_sra, how="outer")
merged_metadata = metadata_masked.drop_duplicates(keep="first")

print(len(headers_seqs))
print(len(merged_metadata))

2448
2224


In [18]:
# Add sequences to a dataframe 
headers_seqs_df = pd.DataFrame.from_dict(headers_seqs, orient="index")
headers_seqs_df = headers_seqs_df.reset_index()
headers_seqs_df["GenBank_Title"] = headers_seqs_df["index"].apply(lambda x: x[1:].split("|")[-1])
headers_seqs_df = headers_seqs_df.rename(columns={0: "Sequence", "index": "Header"})

# Add sequences to the dataframe
metadata_seqs = merged_metadata.merge(headers_seqs_df, on="GenBank_Title", how="left")

print(metadata_seqs)


       Accession GenBank_RefSeq SRA_Accession     BioSample    BioProject  \
0     PV845802.1        GenBank   SRR33681813  SAMN48710861  PRJNA1102327   
1     PV845803.1        GenBank   SRR33681813  SAMN48710861  PRJNA1102327   
2     PV845804.1        GenBank   SRR33681813  SAMN48710861  PRJNA1102327   
3     PV845805.1        GenBank   SRR33681813  SAMN48710861  PRJNA1102327   
4     PV845806.1        GenBank   SRR33681813  SAMN48710861  PRJNA1102327   
...          ...            ...           ...           ...           ...   
2219  PV792621.1        GenBank   SRR33476664  SAMN48398150   PRJNA980729   
2220  PV792622.1        GenBank   SRR33476664  SAMN48398150   PRJNA980729   
2221  PV792623.1        GenBank   SRR33476664  SAMN48398150   PRJNA980729   
2222  PV792624.1        GenBank   SRR33476664  SAMN48398150   PRJNA980729   
2223  PV792625.1        GenBank   SRR33476664  SAMN48398150   PRJNA980729   

          Organism_Name                         Species Genotype  \
0     I

In [19]:
# Create FASTA files per Header

metadata_seqs["Partial_Header"] = metadata_seqs["Header"].apply(lambda x: "|".join(x.split("|")[:-1]))

unique_partial_headers = []

# Each header should repeat 8 times
for partial_header in metadata_seqs["Partial_Header"].values:
    if partial_header not in unique_partial_headers: # Only use the first occurence
        unique_partial_headers.append(partial_header)

print(len(unique_partial_headers)) # Sanity check

278


In [20]:
df_list = []
for partial_header in unique_partial_headers:
    # Get smaller dataframe
    df = metadata_seqs[metadata_seqs["Partial_Header"] == partial_header]
    df = df.sort_values(by="Segment")
    df = df[["Header", "Sequence"]]
    df = df.rename(columns={"Header": "full_header", "Sequence": "sequence"}) # For utils function
    df["full_header"] = df["full_header"].apply(lambda x: ">" + x.split("|")[-1].replace(" ", "_"))
    # Make sure there are 8 segments
    if len(df) == 8:
        df_list.append(df)

In [21]:
print(df_list[0])
print(len(df_list))

                                         full_header  \
0  >Influenza_A_virus_(A/cattle/CA/24-028985-001-...   
1  >Influenza_A_virus_(A/cattle/CA/24-028985-001-...   
2  >Influenza_A_virus_(A/cattle/CA/24-028985-001-...   
3  >Influenza_A_virus_(A/cattle/CA/24-028985-001-...   
4  >Influenza_A_virus_(A/cattle/CA/24-028985-001-...   
5  >Influenza_A_virus_(A/cattle/CA/24-028985-001-...   
6  >Influenza_A_virus_(A/cattle/CA/24-028985-001-...   
7  >Influenza_A_virus_(A/cattle/CA/24-028985-001-...   

                                            sequence  
0  GAATAAAAGAACTGAGAGATCTAATGTCACAGTCTCGCACTCGCGA...  
1  CTCTTCTTGAAAGTTCCAGCGCAAAATGCCATAAGCACCACATTCC...  
2  ATGGAAGACTTTGTGCGACAATGCTTCAATCCAATGATTGTCGAGC...  
3  ATGGAGAACATAGTACTACTTCTTGCAATAGTTAGCCTTGTTAAAA...  
4  ATGGCGTCTCAAGGCACCAAACGATCCTATGAACAAATGGAAACTG...  
5  ATGAATCCAAATCAGAAGATAACAACCATTGGATCAATCTGTATGG...  
6  ATGAGTCTTCTAACCGAGGTCGAAACGTACGTTCTCTCTATCGTCC...  
7  AGTCCCAACACTGTGTTAAGCTTTCAGGTAGACTGCTTTCTTTGGC...  


In [22]:
# Make fasta files. Finally

os.chdir(temp_files)

for df in df_list:
    # print(df)
    file_name = df["full_header"].apply(lambda x: x.split("|")[-1]).values[0] + "_temp.fasta"
    # Forbidden characters
    for c in [">", "/", "|", " ", ":", ",", "(", ")", "-"]:
        file_name = file_name.replace(c, "")
    # print(file_name)
    output_file = open(temp_files + file_name, "w")

    for index, row in df.iterrows():
        name = df.loc[index, "full_header"]
        # print(name)
        sequence = df.loc[index, "sequence"]
        # print(sequence)
        # break 
    # First is header, second is sequence
        output_file.write(name + "\n")
        output_file.write(sequence + "\n")
    output_file.close()

## Re-Labeling Using GenoFlu

In [23]:
# Merging

os.chdir(downloads)

output_genoflu = pd.read_csv("output.tsv", delimiter="\t")

# print(output_genoflu)

# print(metadata_seqs["Header"])

file_name = metadata_seqs["Header"].apply(lambda x: x.split("|")[-1].replace(" ", "_") + "_temp.fasta")
for c in [">", "/", "|", " ", ":", ",", "(", ")", "-"]:
    file_name = file_name.apply(lambda x: x.replace(c, ""))

metadata_seqs["File Name"] = file_name

metadata_genoflu = metadata_seqs.merge(output_genoflu, how="left", on="File Name")

metadata_genoflu = metadata_genoflu.ffill()



print(metadata_genoflu)


       Accession GenBank_RefSeq SRA_Accession     BioSample    BioProject  \
0     PV845802.1        GenBank   SRR33681813  SAMN48710861  PRJNA1102327   
1     PV845803.1        GenBank   SRR33681813  SAMN48710861  PRJNA1102327   
2     PV845804.1        GenBank   SRR33681813  SAMN48710861  PRJNA1102327   
3     PV845805.1        GenBank   SRR33681813  SAMN48710861  PRJNA1102327   
4     PV845806.1        GenBank   SRR33681813  SAMN48710861  PRJNA1102327   
...          ...            ...           ...           ...           ...   
2219  PV792621.1        GenBank   SRR33476664  SAMN48398150   PRJNA980729   
2220  PV792622.1        GenBank   SRR33476664  SAMN48398150   PRJNA980729   
2221  PV792623.1        GenBank   SRR33476664  SAMN48398150   PRJNA980729   
2222  PV792624.1        GenBank   SRR33476664  SAMN48398150   PRJNA980729   
2223  PV792625.1        GenBank   SRR33476664  SAMN48398150   PRJNA980729   

          Organism_Name                         Species Genotype_x  \
0    

We want:
* Host
* Geo-Location
* Isolate
* Year
* Collection Date
* Host Type
* Genotype

In [32]:
# Re-Labeling

os.chdir(home)
animals_ref = pd.read_csv("animals_ref.csv")

# Make sure NaN doesn't mess up the whole name
metadata_genoflu = metadata_genoflu.fillna("")
metadata_genoflu["Host"] = metadata_genoflu["Host"].apply(str.lower)

fix_animals_andersen(metadata_genoflu, animals_ref)

metadata_genoflu["Years"] = metadata_genoflu["Collection_Date"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y"))

# print(metadata_genoflu["Host_Type"])

names = ">" + metadata_genoflu["SRA_Accession"] + "|A/" + metadata_genoflu["Host"] + "/" + metadata_genoflu["Geo_Location"].apply(lambda x: x.split(": ")[-1]) + "/" + metadata_genoflu["Isolate"] + "/" + metadata_genoflu["Years"].apply(lambda x: str(x)) + "|" + metadata_genoflu["Genotype_x"] + "|" + metadata_genoflu["Geo_Location"].apply(lambda x: x.replace(": ", "-")) + "|" + metadata_genoflu["Collection_Date"] + "|" + metadata_genoflu["Host_Type"] + "|" + metadata_genoflu["Genotype_y"]

metadata_genoflu["Name"] = names

print(metadata_genoflu["Name"])


0       >SRR33681813|A/bos taurus/CA/24-028985-001-til...
1       >SRR33681813|A/bos taurus/CA/24-028985-001-til...
2       >SRR33681813|A/bos taurus/CA/24-028985-001-til...
3       >SRR33681813|A/bos taurus/CA/24-028985-001-til...
4       >SRR33681813|A/bos taurus/CA/24-028985-001-til...
                              ...                        
2219    >SRR33476664|A/gallus gallus/NY/25-013275-010-...
2220    >SRR33476664|A/gallus gallus/NY/25-013275-010-...
2221    >SRR33476664|A/gallus gallus/NY/25-013275-010-...
2222    >SRR33476664|A/gallus gallus/NY/25-013275-010-...
2223    >SRR33476664|A/gallus gallus/NY/25-013275-010-...
Name: Name, Length: 2224, dtype: object


## Merge all datasets and de-duplicate again

In [33]:
# # Set up segments

# segments = {1:"PB2", 2:"PB1", 3:"PA", 4:"HA", 5:"NP", 6:"NA", 7:"MP", 8:"NS"}

# metadata_genoflu["Segment_Name"] = metadata_genoflu["Segment"].map(segments)

# # print(metadata_genoflu["Segment_Name"])

# # Separate into several dataframes based on genotype + segment
# segment_genotype_dfs = []
# for segment in segments.values():
#     print(segment)
#     m_g = metadata_genoflu[metadata_genoflu["Segment_Name"] == segment]
#     print(m_g)
#     for genotype in list(set(m_g["Genotype_y"].values)):
#         if "Not assigned:" not in genotype:
#             df = m_g[(m_g["Genotype_y"] == genotype)] # & (metadata_genoflu["Segment"] == segment)]
#             segment_genotype_dfs.append(df)
#             # pair = genotype + "_" + segment
#             # print(pair)

# print(segment_genotype_dfs[3])

# # print(metadata_genoflu["Header"].apply(lambda x: x.split("|")[-1].split("/")[-1].split(")")[2:]))

In [34]:
# Set up segments

segments = {1:"PB2", 2:"PB1", 3:"PA", 4:"HA", 5:"NP", 6:"NA", 7:"MP", 8:"NS"}



metadata_genoflu["Segment_Name"] = metadata_genoflu["Segment"].map(segments)

# print(metadata_genoflu["Segment_Name"])

# Separate into several dataframes based on genotype + segment
# b313_count = 0
segment_genotype_dfs = []
for segment in segments.values():
    # print(segment)
    m_g = metadata_genoflu[metadata_genoflu["Segment_Name"] == segment]
    # print(m_g)
    for genotype in list(set(m_g["Genotype_y"].values)):
        # if "B3.13" in genotype:
        if genotype in genotypes:
            # b313_count += 1
            df = m_g[(m_g["Genotype_y"] == genotype)] # & (metadata_genoflu["Segment"] == segment)]
            # b313_count += len(df)
            segment_genotype_dfs.append(df)
            pair = genotype + "_" + segment
            print(pair)

# print(b313_count)
# print(segment_genotype_dfs[8])

B3.6_PB2
B3.13_PB2
D1.1_PB2
B3.6_PB1
B3.13_PB1
D1.1_PB1
B3.6_PA
B3.13_PA
D1.1_PA
B3.6_HA
B3.13_HA
D1.1_HA
B3.6_NP
B3.13_NP
D1.1_NP
B3.6_NA
B3.13_NA
D1.1_NA
B3.6_MP
B3.13_MP
D1.1_MP
B3.6_NS
B3.13_NS
D1.1_NS


In [35]:
# Create FASTA files

os.chdir(complete_files)

names = []

for df in segment_genotype_dfs:
    if len(df["Genotype_y"].values[0]) > 0:
        file_name = df["Genotype_y"].values[0] + "_" + df["Segment_Name"].values[0] + "_" + date_range + ".fasta"
        output_file = open(complete_files + file_name, "w")

        for index, row in df.iterrows():
            name = df.loc[index, "Name"]
            names.append(name)
            # print(name)
            sequence = df.loc[index, "Sequence"]
            # print(sequence)
            # break 
        # First is header, second is sequence
            output_file.write(name + "\n")
            output_file.write(sequence + "\n")
        output_file.close()

print(len(names))

2216


In [36]:
print(segment_genotype_dfs)

[       Accession GenBank_RefSeq SRA_Accession     BioSample   BioProject  \
1656  PV847514.1        GenBank   SRR33682399  SAMN48711133  PRJNA980729   

          Organism_Name                         Species Genotype_x  \
1656  Influenza A virus  Alphainfluenzavirus influenzae       H5N1   

                     Isolate  Segment  ... Genotype_y  \
1656  25-014787-002-original        1  ...       B3.6   

                            Genotype List Used, >=98.0%  \
1656  PB2:am5, PB1:am4, PA:ea1, HA:ea1, NP:am1.4.1, ...   

                             Genotype Sample Title List  \
1656  am5:23-001855-001:PB2, am4:23-001855-001:PB1, ...   

                            Genotype Percent Match List  \
1656  99.08%, 98.86%, 98.42%, 98.30%, 98.93%, 98.37%...   

              Genotype Mismatch List Genotype Average Depth of Coverage List  \
1656  21, 26, 34, 29, 16, 23, 15, 11       Ran on FASTA - No Coverage Report   

     Host_Type Years                                               Name 

In [37]:
# Concatenate

os.chdir(combined_files)

filenames_gisaid_andersen = []
for genotype in genotypes:
    # print(gisaid_andersen + genotype.replace(".", "_") + "/")
    # for dirpath, dirs, files in os.walk(gisaid_andersen + genotype.replace(".", "_") + "/" + date_range + "_" + genotype.replace(".", "_") + "/"):
    for dirpath, dirs, files in os.walk(gisaid_andersen): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            filenames_gisaid_andersen.append(file_name)
        break 

print(filenames_gisaid_andersen)

filenames_ncbi = []
for dirpath, dirs, files in os.walk(complete_files): # Find the fasta file
    for file in files:
        file_name = os.path.join(dirpath, file) # Get file name
        filenames_ncbi.append(file_name)
    break 

print(filenames_ncbi)

do_not_do_these_partials = []
for ga_file in filenames_gisaid_andersen:
    partial_filename_ga = ga_file.split("_")[-4].split("/")[-1] + "_" + ga_file.split("_")[-3]
    for nv_file in filenames_ncbi:
        partial_filename_nv = nv_file.split("_")[-3].split("/")[-1] + "_" + nv_file.split("_")[-2]
        if partial_filename_ga == partial_filename_nv:
            do_not_do_these_partials.append(partial_filename_ga)
            print(partial_filename_nv)
            filenames = [ga_file, nv_file]
            with open(combined_files + partial_filename_ga + "_combined_" + date_range + ".fasta", 'w') as outfile:
                for fname in filenames:
                    with open(fname) as infile:
                        for line in infile:
                            outfile.write(line)

for ga_file in filenames_gisaid_andersen:
    partial_filename_ga = ga_file.split("_")[-4].split("/")[-1] + "_" + ga_file.split("_")[-3]
    if partial_filename_ga not in do_not_do_these_partials:
        with open(combined_files + partial_filename_ga + "_combined_" + date_range + ".fasta", 'w') as outfile2:
            # for fname in filenames_gisaid_andersen:
            with open(ga_file) as infile1:
                for line in infile1:
                    outfile2.write(line)
    

# lines = 0

# fns_seen = set()
# for filename in filenames_ncbi:
#     print("NCBI_Virus", filename)
#     # lines = 0
#     # filename.split("_")[-5] + "." +   
#     partial_filename = filename.split("_")[-3].split("/")[-1] + "_" + filename.split("_")[-2] 
#     print(partial_filename)
    
#     for fn in filenames_gisaid_andersen:
#         # print(fns_seen)
#         if partial_filename in fn: # and partial_filename not in fns_seen: # If partial filenames match
#             fns_seen.add(fn)
#         #     fns_seen.add(partial_filename)
#             # print("hi", partial_filename)
#             filenames = [filename, fn]
#             with open(combined_files + partial_filename + "_combined_" + update_date + ".fasta", 'w') as outfile:
#                 for fname in filenames:
#                     with open(fname) as infile1:
#                         for line in infile1:
#                             outfile.write(line)

# for fn in filenames_gisaid_andersen: # Extra files
#     if fn not in fns_seen:
#         partial_filename = fn.split("_")[-4].split("/")[-1] + "_" + fn.split("_")[-3] 
#         with open(combined_files + partial_filename + "_combined_" + update_date + ".fasta", 'w') as outfile:
#             for fname in filenames:
#                 with open(fname) as infile1:
#                     for line in infile1:
#                         outfile.write(line)
    
#                             # if line[0] == ">":
#                             #     lines += 1
#             #         infile.close()
#             # outfile.close()
#         # elif partial_filename not in fns_seen: # Save the ones that aren't shared -- if partial filename was not seen before
#         #     # print("hello", partial_filename)
#         #     fns_seen.add(partial_filename)
#         #     with open(combined_files + partial_filename + "_combined_" + update_date + ".fasta", 'w') as outfile2:
#         #         # Copy
#         #         with open(fn) as infile2:
#         #             for line in infile2:
#         #                 outfile2.write(line)
#         # else: # If partial filename doesn't match AND if partial filename has been seen before
#         #     continue 
#         #                 # if line[0] == ">":
#         #                 #     lines += 1
#         #             # infile.close()
#         #         with open(fn) as infile3:
#         #             for line in infile3:
#         #                 outfile2.write(line)
#         #                 # if line[0] == ">":
#                         #     lines += 1
#             #     infile.close()
#         outfile.close()


#     # break 

#     # print(lines)

# # lines = 0
# # Counting
#     with open(combined_files + partial_filename + "_combined_" + update_date + ".fasta", 'r') as outfile:
#         for line in outfile.readlines():
#             if line[0] == ">":
#                 lines += 1

# print(lines)

['C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Combinations/GISAID_Andersen/all_genotypes/06-14-2025--06-27-2025_all_genotypes/A3_HA_combined_06-14-2025--06-27-2025.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Combinations/GISAID_Andersen/all_genotypes/06-14-2025--06-27-2025_all_genotypes/A3_MP_combined_06-14-2025--06-27-2025.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Combinations/GISAID_Andersen/all_genotypes/06-14-2025--06-27-2025_all_genotypes/A3_NA_combined_06-14-2025--06-27-2025.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Combinations/GISAID_Andersen/all_genotypes/06-14-2025--06-27-2025_all_genotypes/A3_NP_combined_06-14-2025--06-27-2025.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Combinations/GISAID_Andersen/all_genotypes/06-14-2025--06-27-2025_all_genotypes/A3_NS_combined_06-14-2025--06-27-2025.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Combinations/GISAID_Andersen/all_genoty

In [38]:
# # Mega fasta file
# all_files = []
# for dirpath, dirs, files in os.walk(combined_files):
#     for file in files:
#         file_name = os.path.join(dirpath, file)
#         all_files.append(file_name)

# with open(combined_files + "all_files_combined_" + update_date + ".fasta", 'w') as outfile3:
#     for fname in all_files:
#         with open(fname) as infile4:
#             for line in infile4:
#                 outfile3.write(line)
# #         infile.close()
# # outfile.close()

In [40]:
# Animals 

os.chdir(home)
animals_ref = pd.read_csv("animals_ref.csv")

# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata_genoflu)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 


['gallus gallus', 'anatidae', 'bos taurus', 'perdicinae', 'meleagris gallopavo']
[]
                      avian               cattle        feline   other_mammal  \
0          great_horned_owl            dairy_cow           cat     deer mouse   
1              common_raven               cattle  domestic_cat    house_mouse   
2             cooper's_hawk  cattle milk product     feral_cat          skunk   
3              coopers_hawk          bovine_milk        feline  striped_skunk   
4                   peafowl              bovine   domestic-cat     norway rat   
..                      ...                  ...           ...            ...   
798  von_schrenck's_bittern                  NaN           NaN            NaN   
799           harris's_hawk                  NaN           NaN            NaN   
800           eurasian_coot                  NaN           NaN            NaN   
801                   layer                  NaN           NaN            NaN   
802              perdicin

In [ ]:
# # Concat

# os.chdir(combined_files)

# for dirpath, dirs, files in os.walk(complete_files):
#     for file in files:
#         file_name = os.path.join(dirpath, file)
#         print(file_name)
#         for dirpath1, dirs1, files1 in os.walk(gisaid_andersen):
#             for file1 in files1:
#                 file_name1 = os.path.join(dirpath1, file1)
#                 if file_name.split("/")[-1].split("_")[0] + file_name.split("/")[-1].split("_")[1] == file_name1.split("/")[-1].split("_")[0] + file_name1.split("/")[-1].split("_")[1]:
                    
#                     output_path = combined_files + file_name1.split("/")[-1] # Genotype and Segment should all be the same
#                     output_file = open(output_path, "w")
#                     with open(file_name) as f:
#                         for line in f.readlines():
#                             output_file.write(line)
#                             # output_file.write("\n")
#                         f.close()
#                     with open(file_name1) as f1:
#                        for line in f1.readlines():
#                             output_file.write(line)
#                             # output_file.write("\n")
#                     f1.close()  

#                     output_file.close()